<a href="https://colab.research.google.com/github/quain7/stm32-project/blob/quain7-patch-3/RQT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal
import scipy.interpolate
import gpxpy
import gpxpy.geo
import pywt
import folium
from hmmlearn import hmm
from ahrs.filters import Madgwick
from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler
import requests
from scipy.spatial import KDTree

def q2R(q):
    q0, q1, q2, q3 = q
    return np.array([
        [1 - 2*q2**2 - 2*q3**2, 2*q1*q2 - 2*q0*q3, 2*q1*q3 + 2*q0*q2],
        [2*q1*q2 + 2*q0*q3, 1 - 2*q1**2 - 2*q3**2, 2*q2*q3 - 2*q0*q1],
        [2*q1*q3 - 2*q0*q2, 2*q2*q3 + 2*q0*q1, 1 - 2*q1**2 - 2*q2**2]
    ])

def main():

    # 1. Data Loading
    print("Loading datasets...")

    imu_columns = ['timestamp_ms', 'ax', 'ay', 'az', 'gx', 'gy', 'gz']
    try:
        df_imu = pd.read_csv('ROUTE009.CSV', names=imu_columns)

        df_imu = df_imu.apply(pd.to_numeric, errors='coerce').dropna().reset_index(drop=True)

    except FileNotFoundError:
        print("Warning: TEST (2).TXT not found. Please ensure the file exists.")
        return

    # Convert LSB to physical units (±2G & ±250dps)
    df_imu[['ax', 'ay', 'az']] = df_imu[['ax', 'ay', 'az']] / 16384.0
    dps_to_rads = np.pi / 180.0
    df_imu[['gx', 'gy', 'gz']] = (df_imu[['gx', 'gy', 'gz']] / 131.0) * dps_to_rads

    try:
        with open('20260707-180605.gpx', 'r') as gpx_file:
            gpx = gpxpy.parse(gpx_file)

        gps_data = []
        for track in gpx.tracks:
            for segment in track.segments:
                for point in segment.points:
                    gps_data.append({
                        'time': point.time,
                        'lat': point.latitude,
                        'lon': point.longitude,
                        'ele': point.elevation
                    })
        df_gps = pd.DataFrame(gps_data)
        df_gps['time'] = pd.to_datetime(df_gps['time'], utc=True)
    except FileNotFoundError:
        print("Warning: route.gpx not found. Please ensure the file exists.")
        return

    # 2. Time Synchronization

    print("Synchronizing timeframes...")

    accel_mag = np.sqrt(df_imu['ax']**2 + df_imu['ay']**2 + df_imu['az']**2)
    rolling_var = accel_mag.rolling(window=100, min_periods=10).var()

    noise_threshold = 0.005
    movement_mask = rolling_var > noise_threshold

    if not movement_mask.any():
        raise ValueError("No movement detected in IMU data to establish anchor.")

    imu_start_idx = movement_mask.idxmax()
    imu_anchor_ms = df_imu.loc[imu_start_idx, 'timestamp_ms']

    # --- GPS Anchor ---
    distances = [0.0]
    for i in range(1, len(df_gps)):
        p1 = gpxpy.geo.Location(df_gps.loc[i-1, 'lat'], df_gps.loc[i-1, 'lon'])
        p2 = gpxpy.geo.Location(df_gps.loc[i, 'lat'], df_gps.loc[i, 'lon'])
        distances.append(p1.distance_2d(p2))

    df_gps['dist'] = distances
    df_gps['dt'] = df_gps['time'].diff().dt.total_seconds().fillna(0)
    df_gps['speed'] = np.where(df_gps['dt'] > 0, df_gps['dist'] / df_gps['dt'], 0)

    speed_threshold = 1.0 # m/s
    gps_movement_mask = df_gps['speed'] > speed_threshold
    if not gps_movement_mask.any():
        raise ValueError("No movement detected in GPS data to establish anchor.")

    gps_start_idx = gps_movement_mask.idxmax()
    gps_anchor_time = df_gps.loc[gps_start_idx, 'time']

    # --- Alignment & Interpolation ---
    base_timestamp = gps_anchor_time - pd.to_timedelta(imu_anchor_ms, unit='ms')
    df_imu['abs_time'] = base_timestamp + pd.to_timedelta(df_imu['timestamp_ms'], unit='ms')

    df_gps['time_sec'] = df_gps['time'].astype(np.int64) / 1e9
    df_imu['time_sec'] = df_imu['abs_time'].astype(np.int64) / 1e9

    df_gps_clean = df_gps.drop_duplicates(subset=['time_sec']).sort_values('time_sec')

    interp_lat = scipy.interpolate.interp1d(
        df_gps_clean['time_sec'], df_gps_clean['lat'],
        kind='linear', bounds_error=False,
        fill_value=(df_gps_clean['lat'].iloc[0], df_gps_clean['lat'].iloc[-1])
    )
    interp_lon = scipy.interpolate.interp1d(
        df_gps_clean['time_sec'], df_gps_clean['lon'],
        kind='linear', bounds_error=False,
        fill_value=(df_gps_clean['lon'].iloc[0], df_gps_clean['lon'].iloc[-1])
    )

    df_imu['lat'] = interp_lat(df_imu['time_sec'])
    df_imu['lon'] = interp_lon(df_imu['time_sec'])


    # 3. Calibration

    print("Calibrating and running Madgwick AHRS...")

    stat_mask = df_imu['timestamp_ms'] < imu_anchor_ms

    gx_bias = df_imu.loc[stat_mask, 'gx'].mean()
    gy_bias = df_imu.loc[stat_mask, 'gy'].mean()
    gz_bias = df_imu.loc[stat_mask, 'gz'].mean()

    ax_bias = df_imu.loc[stat_mask, 'ax'].mean()
    ay_bias = df_imu.loc[stat_mask, 'ay'].mean()
    az_bias = df_imu.loc[stat_mask, 'az'].mean() - 1.0

    df_imu['gx'] -= gx_bias
    df_imu['gy'] -= gy_bias
    df_imu['gz'] -= gz_bias
    df_imu['ax'] -= ax_bias
    df_imu['ay'] -= ay_bias
    df_imu['az'] -= az_bias

    dt_ahrs = 1.0 / 100.0 # 100 Hz


    madgwick = Madgwick(frequency=100.0)

    Q = np.zeros((len(df_imu), 4))
    Q[0] = [1.0, 0.0, 0.0, 0.0]

    gyr_data = df_imu[['gx', 'gy', 'gz']].values
    acc_data = df_imu[['ax', 'ay', 'az']].values

    for i in range(1, len(df_imu)):

        Q[i] = madgwick.updateIMU(Q[i-1], gyr_data[i], acc_data[i])

    a_global = np.zeros((len(df_imu), 3))
    for i in range(len(df_imu)):
        R = q2R(Q[i])
        a_global[i] = R @ acc_data[i]

    dyn_az = a_global[:, 2] - 1.0
    df_imu['dyn_az'] = dyn_az


        # =========================================================================
    # 4-6. Two-Stream DSP Pipeline (Hard Filter + HMM Texture)
    # =========================================================================

    print("Applying zero-phase High-Pass filter to strictly remove gravity leakage...")
    # 1. Використовуємо sosfiltfilt для двонаправленої фільтрації (нульовий фазовий зсув)
    sos_hp = scipy.signal.butter(4, 0.5, 'hp', fs=100, output='sos')
    az_filtered = scipy.signal.sosfiltfilt(sos_hp, df_imu['az'].fillna(0).values)

    print("Extracting features using 50% overlapping windows...")
    window_size = 100
    step_size = 50

    features_hmm = []
    is_pothole_list = []
    chunk_indices = []

    # 2. Ітерація ковзним вікном з 50% перекриттям
    for i in range(0, len(az_filtered) - window_size + 1, step_size):
        window = az_filtered[i:i+window_size]

        # --- Потік 1 (Жорсткий фільтр аномалій) ---
        max_impact = np.max(np.abs(window))
        kurt = kurtosis(window, fisher=True)
        if np.isnan(kurt): kurt = 0

        # Override Rule: Якщо удар > 1.5G або куртозис > 4.0 -> це гарантовано яма
        if max_impact > 1.5 or kurt > 4.0:
            is_pothole_list.append(1)
        else:
            is_pothole_list.append(0)

        # --- Потік 2 (HMM для загальної текстури дороги) ---
        freqs, psd = scipy.signal.welch(window, fs=100, nperseg=window_size)

        band_low = np.sum(psd[(freqs >= 0.5) & (freqs < 3.0)])
        band_mid = np.sum(psd[(freqs >= 3.0) & (freqs < 10.0)])
        band_high = np.sum(psd[(freqs >= 10.0) & (freqs <= 40.0)])

        total_e = band_low + band_mid + band_high

        # Зберігаємо енергію по смугах (і total_e для майбутнього сортування кластерів)
        features_hmm.append([band_low, band_mid, band_high, total_e])

        # Беремо координати по ЦЕНТРУ вікна для максимальної точності на карті
        chunk_indices.append(i + window_size // 2)

    features_hmm = np.array(features_hmm)

    # Формуємо базовий датафрейм
    df_windows = pd.DataFrame({
        'lat': df_imu['lat'].iloc[chunk_indices].values,
        'lon': df_imu['lon'].iloc[chunk_indices].values,
        'time_sec': df_imu['time_sec'].iloc[chunk_indices].values,
        'is_pothole': is_pothole_list
    })

    print("Training Gaussian HMM on spectral features (3 components)...")
    # Передаємо в HMM лише спектральні фічі (перші 3 колонки)
    X_train = features_hmm[:, :3]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    # Навчаємо модель розрізняти лише 3 базові типи покриття
    model = hmm.GaussianHMM(n_components=3, covariance_type="diag", n_iter=100, random_state=42)
    model.fit(X_scaled)
    hidden_states = model.predict(X_scaled)

    print("Logically identifying texture states based on energy...")
    cluster_energies = []
    for state in range(3):
        mask = (hidden_states == state)
        if np.sum(mask) > 0:
            # Використовуємо 4-ту колонку (total_e) для визначення жорсткості
            mean_energy = np.mean(features_hmm[mask, 3])
        else:
            mean_energy = 0
        cluster_energies.append(mean_energy)

    sorted_states = np.argsort(cluster_energies)

    # Класифікуємо від найгладкішого до найжорсткішого
    texture_labels_map = {
        sorted_states[0]: 'Smooth',
        sorted_states[1]: 'Worn/Rough',
        sorted_states[2]: 'Cobblestone'
    }

    # --- Зливання Потоків (Override Rule) ---
    print("Merging Stream 1 (Anomalies) and Stream 2 (Texture)...")

    final_labels = []
    for i in range(len(df_windows)):
        if df_windows['is_pothole'].iloc[i] == 1:
            # Примусовий статус "Яма" незалежно від рішення HMM
            final_labels.append('Pothole')
        else:
            # Інакше застосовуємо текстуру від HMM
            final_labels.append(texture_labels_map[hidden_states[i]])

    df_windows['label'] = final_labels

    print("\n--- TWO-STREAM PIPELINE SUMMARY ---")
    print(df_windows['label'].value_counts())
    print("--------------------------------\n")

    # =========================================================================
    # 7. Visualization of Raw GPS Track (No APIs)
    # =========================================================================
    import folium

    print("Generating Folium map from raw GPS data...")

    start_lat = df_windows['lat'].iloc[0]
    start_lon = df_windows['lon'].iloc[0]
    m = folium.Map(location=[start_lat, start_lon], zoom_start=15, tiles='CartoDB positron')

    # Палітра кольорів під нашу Two-Stream архітектуру
    color_map = {
        'Smooth': '#00FF00',      # Зелений
        'Worn/Rough': '#ADFF2F',  # Салатовий
        'Cobblestone': '#FFA500', # Помаранчевий
        'Pothole': '#FF0000'      # Червоний
    }

    # Малюємо лінію прямо по ваших GPS координатах
    for i in range(len(df_windows) - 1):
        p1 = [df_windows['lat'].iloc[i], df_windows['lon'].iloc[i]]
        p2 = [df_windows['lat'].iloc[i+1], df_windows['lon'].iloc[i+1]]

        label = df_windows['label'].iloc[i]

        folium.PolyLine(
            [p1, p2],
            color=color_map[label],
            weight=6,
            opacity=0.8
        ).add_to(m)

    m.save('road_quality_map.html')
    print("Pipeline complete! Reliable raw map saved to 'road_quality_map.html'.")

if __name__ == "__main__":
    main()

Прив'язка координат до дороги через OSRM API (Map Matching)...


NameError: name 'df_windows' is not defined

In [ ]:
pip install numpy pandas scipy gpxpy PyWavelets folium hmmlearn ahrs requests


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.7/244.7 kB 10.7 MB/s eta 0:00:00
